# 1. Environment Setup
Install Required Libraries:

In [ ]:
# 1a. Environment Setup.
!pip install datasets
!pip -q install langchain
!pip install pypdf2
!pip install sentence-transformers
!pip install transformers
!pip install xformers
!pip install faiss-cpu
!pip install faiss-gpu
!pip install nltk
!pip install gradio
!pip install datasets
!pip -q install --upgrade huggingface_hub
!pip -q install langchain_community
!pip -q install langchain_huggingface
!pip install transformers accelerate sentence-transformers scikit-learn
!pip install requests
!pip install simplejson
!pip install json
!pip uninstall -y torch torchvision transformers sentence-transformers
!pip cache purge
!pip install --no-cache-dir torch torchvision transformers sentence-transformers
!pip install --upgrade --quiet huggingface_hub
!pip install groq
!pip install openai
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1b. Installing required libraries.
import torch
torch.cuda.empty_cache()
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,root_mean_squared_error
import matplotlib.pyplot as plt
from datasets import load_dataset
from langchain.llms import HuggingFaceEndpoint
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from sklearn.feature_extraction.text import CountVectorizer
import os
from groq import Groq
from getpass import getpass
import numpy as np
import faiss
import gradio as gr
import time
import requests
import json
import torch
import datasets
import nltk
nltk.download('punkt_tab')
import torch
import torchvision
import datasets
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
import datasets
from datasets import load_dataset, load_from_disk, DatasetDict
import tempfile
import statistics
import csv
import random
from huggingface_hub import InferenceClient
from langchain.embeddings import OpenAIEmbeddings
import csv
import warnings
from sentence_transformers import SentenceTransformer, util
from torch.cuda.amp import autocast
import torch
torch.cuda.empty_cache()
torch.set_default_dtype(torch.float16)  # Mixed precision
torch.backends.cuda.matmul.allow_tf32 = True  # Tensor cores for efficiency
import re
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, roc_auc_score, f1_score, precision_score, accuracy_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# 1c. Connect to HuggingFace with access token.
pass_token = getpass("Enter your HuggingFace access token: ")
os.environ["HF_TOKEN"] = pass_token
del pass_token

Enter your HuggingFace access token: ··········


In [ ]:
# 1d. Connect to GROQ with access token.
pass_token = getpass("Enter your GROQ access token: ")
os.environ["GROQ_TOKEN"] = pass_token
del pass_token

Enter your GROQ access token: ··········


# 2.Load the RAGBench Dataset
Code to Load and Prepare Dataset:

In [ ]:

# 2. Load datasets from huggingface.
def load_ragbench(local_path="ragbench"):
    """
    Load the ragbench dataset from local storage if available,
    otherwise download from Hugging Face and save locally.

    Returns:
        dict: A dictionary containing train, validation, and test splits for each dataset.
    """
    dataset_names = ['emanual','expertqa', 'cuad', 'covidqa','delucionqa','techqa'
                     'finqa','hagrid','hotpotqa','msmarco','tatqa','pubmedqa']
    data_splits = {"train": {}, "validation": {}, "test": {}}
    ragbench = {}

    # Check if datasets are available locally.
    if os.path.exists(local_path):
        print(f"Loading datasets from local storage: {local_path}...")
        for dataset_name in dataset_names:
            dataset_path = os.path.join(local_path, dataset_name)
            if os.path.exists(dataset_path):
                ragbench[dataset_name] = load_from_disk(dataset_path)
                data_splits["train"][dataset_name] = ragbench[dataset_name]["train"]
                data_splits["validation"][dataset_name] = ragbench[dataset_name].get("validation", None)
                data_splits["test"][dataset_name] = ragbench[dataset_name].get("test", None)
                print(f"Loaded {dataset_name} from local storage.")
    else:
        print("Downloading datasets from Hugging Face...")
        os.makedirs(local_path, exist_ok=True)

        for dataset_name in dataset_names:
            try:
                ragbench[dataset_name] = load_dataset("rungalileo/ragbench", dataset_name)
                print(f"Downloaded dataset: {dataset_name}")

                # Store different splits
                data_splits["train"][dataset_name] = ragbench[dataset_name]["train"]
                data_splits["validation"][dataset_name] = ragbench[dataset_name].get("validation", None)
                data_splits["test"][dataset_name] = ragbench[dataset_name].get("test", None)

                # Save dataset locally.
                dataset_path = os.path.join(local_path, dataset_name)
                ragbench[dataset_name].save_to_disk(dataset_path)
                print(f"Saved {dataset_name} to {dataset_path}")

            except Exception as e:
                print(f"Failed to load {dataset_name}: {e}")

    return data_splits

# Load or download datasets.
data_splits = load_ragbench()

# Combine all datasets for each split
combined_data = {}
for split in ["train", "validation", "test"]:
    split_data = [data_splits[split][ds] for ds in data_splits[split] if data_splits[split][ds] is not None]
    if split_data:
        combined_data[split] = datasets.concatenate_datasets(split_data)
        print(f"Combined {split.capitalize()} Dataset: {combined_data[split]}")
    else:
        print(f"No {split} datasets were loaded.")


documents = combined_data["documents"] if "documents" in combined_data else None
questions = combined_data["question"] if "question" in combined_data else None
response = combined_data["response"] if "response" in combined_data else None


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/288k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/305k [00:00<?, ?B/s]

...

Downloaded dataset: pubmedqa


Saving the dataset (0/1 shards):   0%|          | 0/19600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2450 [00:00<?, ? examples/s]

Saved pubmedqa to ragbench/pubmedqa
Combined Train Dataset: Dataset({
    features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score'],
    num_rows: 59592
})
Combined Validation Dataset: Dataset({
    features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'ove

# 3. Split Documents into Chunks
Chunking Code:

In [ ]:

# 3. Combining the documents to individual sentences then using Sliding Window Chunking technique.
def split_into_chunks(doc, window_size=4, overlap=1):
    """
    Splits a text into sentence-based chunks with sliding window overlap.

    Args:
        text (str): Input text to split.
        window_size (int): Number of sentences per chunk.
        overlap (int): Number of overlapping sentences between chunks.

    Returns:
        list: List of sentence chunks.
    """
    sentences = sent_tokenize(doc)
    chunks = []
    for i in range(0, len(sentences), window_size - overlap):
        chunk = sentences[i:i + window_size]
        chunks.append(" ".join(chunk))
    return chunks


# Flatten and tokenize documents into list/nested lists.
documents_list = []

# Check if documents is not None and in the expected nested list format.
# If not, process combined_data["train"]["documents"] instead.
if documents is None or not (isinstance(documents, list) and all(
        isinstance(sublist, list) and all(isinstance(sentence, str) for sentence in sublist)
        for sublist in documents
)):
    documents = combined_data["train"]["documents"]

# Flatten the nested list into a single list of sentences and join the sentences.
sentences = [sentence for sublist in documents for sentence in sublist]
doc = '. '.join(sentences)

chunks = split_into_chunks(doc)


documents_list.extend([Document(page_content=chunk) for chunk in chunks])
chunked_documents = [doc.page_content for doc in documents_list]
print("Total number of chunks: ", len(chunked_documents))

Total number of chunks:  367724


# 4. Create Embeddings and Build a Vector Database
Embedding Creation and FAISS Index:

In [ ]:

# 4a. Selecting an Embedding Model for Vector Embeddings.
def load_embedding_model(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    # "sentence-transformers/all-MiniLM-L6-v2" (384D), BAAI/bge-base-en (768D)
    """Load and return the embedding model."""
    return SentenceTransformer(model_name)



# 4b. Compute the embeddings for any given text.
def get_embedding(query, embedding_model):
    """Converts the input text to vector embeddings."""
    # Ensure query is a string, if it's a list, joins it's elements
    if isinstance(query, list):
        query = " ".join(query)
    # Check if query is not a string and convert to a string.
    elif not isinstance(query, str):
        query = str(query)

    query_embedding = embedding_model.encode([query])
    return np.array(query_embedding).reshape(1, -1)



# 4c. Function to handle the generation, loading, and saving of embeddings, as well as building the Vector DB and mapping metadata.
def create_or_load_vector_database(chunked_documents, embeddings_filename="embeddings.npy"):
    """Generate or load embeddings (if available in the local), build FAISS index, and map metadata."""
    embedding_model = load_embedding_model()

    # Load embeddings if the file exists, otherwise generate and save new embeddings.
    if os.path.exists(embeddings_filename):
        print(f"Loading embeddings from {embeddings_filename}...")
        corpus_embeddings = np.load(embeddings_filename)
    else:
        print("Generating new embeddings")
        corpus_embeddings = embedding_model.encode([chunk for chunk in chunked_documents])
        np.save(embeddings_filename, corpus_embeddings)
        print(f"Embeddings saved to {embeddings_filename}")

    # Build FAISS index.
    faiss_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
    faiss_index.add(corpus_embeddings)
    print(f"Total vectors in FAISS index: {faiss_index.ntotal}")

    # Map the chunk metadata for retrieval.
    metadata = {i: chunk for i, chunk in enumerate(chunked_documents)}
    print("Vector database successfully created!")

    return faiss_index, metadata , corpus_embeddings


faiss_index, metadata, corpus_embeddings = create_or_load_vector_database(chunked_documents)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading embeddings from embeddings.npy...
Total vectors in FAISS index: 355246
Vector database successfully created!


# 5. Perform Retrieval and Generate Responses
Define Retrieval and Generation Functions:

In [ ]:


# 5a. Using Cosine Similarity Index on the KNN documents retrieved and extracts top_k.
def create_cosine_similarity_index_test(query_embedding, faiss_index, corpus_embeddings, k_nearest_neighbors=1000):
    """
    Uses FAISS to retrieve the top-k nearest neighbors based on inner product.
    Computes cosine similarities only for these retrieved candidates.
    Returns the cosine similarity scores and indices.
    """
    # Ensure query embedding is 2D and normalized
    query_embedding = query_embedding.reshape(1, -1)
    query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True)  # Normalize query embedding

    # Retrieve k-nearest neighbors using FAISS (fast search)
    distances, indices = faiss_index.search(query_embedding, k_nearest_neighbors)

    # Normalize retrieved embeddings (if not already normalized in the FAISS index)
    retrieved_embeddings = corpus_embeddings[indices[0]]
    retrieved_embeddings /= np.linalg.norm(retrieved_embeddings, axis=1, keepdims=True)

    # Compute cosine similarity
    cosine_sim_matrix = np.dot(query_embedding, retrieved_embeddings.T)[0]

    return cosine_sim_matrix, indices[0]



# 5b. Main function for performing query embedding, corpus embedding, and MMR Retrieval.
def mmr_search_test(query, corpus_embeddings, faiss_index, embedding_model, top_k=3, diversity=0.8, k_nearest_neighbors=1000):
    """
    Maximal Marginal Relevance (MMR) search to select diverse documents.
    - First, retrieve top-k using FAISS.
    - Then, apply MMR on this retrieved subset.
    """

    # Generate query embedding and normalize
    query_embedding = get_embedding(query, embedding_model)
    query_embedding /= np.linalg.norm(query_embedding)  # Normalize query embedding

    # Retrieve nearest neighbors (FAISS fast lookup)
    cosine_sim_matrix, top_k_indices = create_cosine_similarity_index_test(
        query_embedding, faiss_index, corpus_embeddings, k_nearest_neighbors
    )

    # Convert indices to NumPy array for faster operations
    top_k_indices = np.array(top_k_indices)

    # Dictionary lookup for cosine similarity scores
    similarity_lookup = {idx: sim for idx, sim in zip(top_k_indices, cosine_sim_matrix)}

    # MMR selection
    selected = []
    candidate_indices = set(top_k_indices)  # Use a set for O(1) removals

    while len(selected) < top_k and candidate_indices:
        if selected:
            selected_embeddings = corpus_embeddings[selected]  # Get already selected embeddings
            selected_embeddings /= np.linalg.norm(selected_embeddings, axis=1, keepdims=True)  # Normalize

            # Compute diversity scores in a vectorized manner
            max_diversity_scores = np.max(
                cosine_similarity(corpus_embeddings[list(candidate_indices)], selected_embeddings), axis=1
            )
        else:
            max_diversity_scores = np.zeros(len(candidate_indices))

        # Compute MMR scores
        candidate_indices_list = list(candidate_indices)
        mmr_scores = [similarity_lookup[idx] - diversity * div_score for idx, div_score in zip(candidate_indices_list, max_diversity_scores)]

        # Select the document with the highest MMR score
        best_doc = candidate_indices_list[np.argmax(mmr_scores)]
        selected.append(best_doc)
        candidate_indices.remove(best_doc)

    return selected


In [ ]:

# 5a. Using FAISS with IVF (Inverted File Index) for Approximate Nearest Neighbor Search
def create_cosine_similarity_index(query_embedding, faiss_index, corpus_embeddings, k_nearest_neighbors=1000):
    """
    Uses FAISS to retrieve the top-k nearest neighbors based on inner product.
    Computes cosine similarities only for these retrieved candidates.
    Returns the cosine similarity scores and indices.
    """
    # Ensure query embedding is 2D and normalized
    query_embedding = query_embedding.reshape(1, -1)
    query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True)  # Normalize query embedding

    # Retrieve k-nearest neighbors using FAISS (approximate search with IVF)
    distances, indices = faiss_index.search(query_embedding, k_nearest_neighbors)

    # Normalize retrieved embeddings (if not already normalized in the FAISS index)
    retrieved_embeddings = corpus_embeddings[indices[0]]
    retrieved_embeddings /= np.linalg.norm(retrieved_embeddings, axis=1, keepdims=True)

    # Compute cosine similarity for retrieved candidates
    cosine_sim_matrix = np.dot(query_embedding, retrieved_embeddings.T)[0]

    return cosine_sim_matrix, indices[0]

# 5b. Main function for performing query embedding, corpus embedding, and MMR Retrieval.
def mmr_search(query, corpus_embeddings, faiss_index, embedding_model, top_k=3, diversity=0.8, k_nearest_neighbors=1000):
    """
    Maximal Marginal Relevance (MMR) search to select diverse documents.
    - First, retrieve top-k using FAISS.
    - Then, apply MMR on this retrieved subset.
    """

    # Generate query embedding and normalize
    query_embedding = get_embedding(query, embedding_model)
    query_embedding /= np.linalg.norm(query_embedding)  # Normalize query embedding

    # Retrieve nearest neighbors (FAISS fast lookup with approximate search)
    cosine_sim_matrix, top_k_indices = create_cosine_similarity_index(
        query_embedding, faiss_index, corpus_embeddings, k_nearest_neighbors
    )

    # Convert indices to NumPy array for faster operations
    top_k_indices = np.array(top_k_indices)

    # Dictionary lookup for cosine similarity scores
    similarity_lookup = {idx: sim for idx, sim in zip(top_k_indices, cosine_sim_matrix)}

    # Select a smaller subset for MMR based on top-k nearest neighbors
    candidate_indices = set(top_k_indices)  # Use a set for O(1) removals

    # Precompute the cosine similarities of the candidate set
    candidate_embeddings = corpus_embeddings[list(candidate_indices)]
    candidate_embeddings /= np.linalg.norm(candidate_embeddings, axis=1, keepdims=True)

    # MMR selection
    selected = []

    while len(selected) < top_k and candidate_indices:
        if selected:
            # Get embeddings of already selected documents
            selected_embeddings = corpus_embeddings[selected]
            selected_embeddings /= np.linalg.norm(selected_embeddings, axis=1, keepdims=True)

            # Compute diversity scores (cosine similarity to already selected documents)
            diversity_scores = cosine_similarity(candidate_embeddings, selected_embeddings).max(axis=1)
        else:
            diversity_scores = np.zeros(len(candidate_indices))

        # Compute MMR scores: similarity - diversity penalty
        mmr_scores = [similarity_lookup[idx] - diversity * div_score for idx, div_score in zip(candidate_indices, diversity_scores)]

        # Select the document with the highest MMR score
        best_doc = max(candidate_indices, key=lambda idx: mmr_scores[list(candidate_indices).index(idx)])
        selected.append(best_doc)
        candidate_indices.remove(best_doc)

    return selected


In [ ]:


def convert_chunks_to_dict(chunks):
    """
    Converts a list of context chunks into a dictionary with keys following the pattern
    ["0a.", "0b.", ..., "1a.", "1b.", ...].

    Sentences in each chunk are split using common delimiters like `.`, `!`, and `?`.

    Parameters:
    - chunks (list of str): A list where each element is a chunk containing multiple sentences.

    Returns:
    - dict: A dictionary where keys follow the specified pattern and values are individual sentences.
    """
    result = {}
    for chunk_index, chunk in enumerate(chunks):
        # Split the chunk into sentences using common sentence delimiters
        sentences = re.split(r'(?<=[.!?])\s+', chunk.strip())
        for sentence_index, sentence in enumerate(sentences):
            if sentence.strip():
                key = f"{chunk_index}{chr(97 + sentence_index)}."
                result[key] = sentence.strip()
    return result



In [ ]:


# 5b. Using an LLM model to generate a response based on the context chunks obtained from MMR Retrieval
def generate_response(document, question, max_new_tokens=512):

    # Initialize the HuggingFaceEndpoint with your desired parameters
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Meta-Llama-3-70B-Instruct", # "HuggingFaceH4/zephyr-7b-beta",  #"meta-llama/Llama-3.1-8B-Instruct",
        task="text-generation",
        max_new_tokens=max_new_tokens,
        huggingfacehub_api_token=os.environ["HF_TOKEN"],
        top_k=30,
        temperature=0.5,
        repetition_penalty=1.03,)

    prompt=f"""
    You are a helpful assitant providing answers to user queries.
    Answer the question using the provided document only.
    Choose the relevant information from the provided document that is helpful in answering the question only.
    Context: {document}
    Question: {question}

    """

    response = llm.invoke(prompt)
    match = re.search(r'\{"(.*)"\}', response)
    if match:
        extracted_string = match.group(1)
        return extracted_string
    else:
        return response




def convert_responses_to_dict(response_text):
    """
    Converts a response text into a dictionary where each sentence is assigned
    a key following the pattern ['a.', 'b.', 'c.', ...].

    Parameters:
    - response_text (str): The input text containing multiple sentences.

    Returns:
    - dict: A dictionary where keys follow the specified pattern and values are individual sentences.
    """
    result = {}
    # Split the response into sentences using common sentence delimiters.
    sentences = re.split(r'(?<=[.!?])\s+', response_text.strip())

    for index, sentence in enumerate(sentences):
        if sentence.strip():
            key = f"{chr(97 + index)}."
            result[key] = sentence.strip()

    return result



# 6. Evaluate the System
Attribute Extraction and Metrics:

In [ ]:

from groq import Groq

def generate_trace_metrics(documents, question, answer):
    """
    Generate trace metrics using Groq API with Llama-3.3-70B-Versatile.
    """
    client = Groq(api_key=os.environ["GROQ_TOKEN"])

    prompt = f"""
    I asked someone to answer a question based on one or more documents.
    Your task is to review their response and assess whether or not each sentence
    in that response is supported by text in the documents. And if so, which
    sentences in the documents provide that support. You will also tell me which
    of the documents contain useful information for answering the question, and
    which of the documents the answer was sourced from.
    Here are the documents, each of which is split into sentences. Alongside each
    sentence is associated key, such as ’0a.’ or ’0b.’ that you can use to refer
    to it:
    ‘‘‘
    {documents}
    ‘‘‘
    The question was:
    ‘‘‘
    {question}
    ‘‘‘
    Here is their response, split into sentences. Alongside each sentence is
    associated key, such as ’a.’ or ’b.’ or 'c.' that you can use to refer to it. Note
    that these keys are unique to the response, and are not related to the keys
    in the documents:
    ‘‘‘
    {answer}
    ‘‘‘

    You must respond with a JSON object matching this schema:
    ‘‘‘
    {{
      "relevance_explanation": string,
      "all_relevant_sentence_keys": [string],
      "overall_supported_explanation": string,
      "overall_supported": boolean,
      "sentence_support_information":
      [
       {{
         "response_sentence_key": string,
         "explanation": string,
         "supporting_sentence_keys": [string],
         "fully_supported": boolean
       }},
    ],
    "all_utilized_sentence_keys": [string]
    }}
    ‘‘‘

    The relevance_explanation field is a string explaining which documents
    contain useful information for answering the question. Provide a step-by-step
    breakdown of information provided in the documents and how it is useful for
    answering the question.

    The all_relevant_sentence_keys field is a list of all document sentences keys
    (e.g. ’0a.’,'0b.') that are revant to the question. Include every sentence that is
    useful and relevant to the question, even if it was not used in the response,
    or if only parts of the sentence are useful. Ignore the provided response when
    making this judgement and base your judgement solely on the provided documents
    and question. Omit sentences that, if removed from the document, would not
    impact someone’s ability to answer the question.

    The overall_supported_explanation field is a string explaining why the response
    *as a whole* is or is not supported by the documents. In this field, provide a
    step-by-step breakdown of the claims made in the response and the support (or
    lack thereof) for those claims in the documents. Begin by assessing each claim
    separately, one by one; don’t make any remarks about the response as a whole
    until you have assessed all the claims in isolation.

    The overall_supported field is a boolean indicating whether the response as a
    whole is supported by the documents. This value should reflect the conclusion
    you drew at the end of your step-by-step breakdown in overall_supported_explanation.

    In the sentence_support_information field, provide information about the support
    *for each sentence* in the response.

    The sentence_support_information field is a list of objects, one for each sentence
    in the response. Each object MUST have the following fields:
    - response_sentence_key: a string identifying the sentence in the response.
    This key is the same as the one used in the response above.
    - explanation: a string explaining why the sentence is or is not supported by the
    documents.
    - supporting_sentence_keys: keys (e.g. ’0a.’) of sentences from the documents that
    support the response sentence. If the sentence is not supported, this list MUST
    be empty. If the sentence is supported, this list MUST contain one or more keys.
    In special cases where the sentence is supported, but not by any specific sentence,
    you can use the string "supported_without_sentence" to indicate that the sentence
    is generally supported by the documents. Consider cases where the sentence is
    expressing inability to answer the question due to lack of relevant information in
    the provided contex as "supported_without_sentence". In cases where the sentence
    is making a general statement (e.g. outlining the steps to produce an answer, or
    summarizing previously stated sentences, or a transition sentence), use the
    sting "general".In cases where the sentence is correctly stating a well-known fact,
    like a mathematical formula, use the string "well_known_fact". In cases where the
    sentence is performing numerical reasoning (e.g. addition, multiplication), use
    the string "numerical_reasoning".
    - fully_supported: a boolean indicating whether the sentence is fully supported by
    the documents.
    - This value should reflect the conclusion you drew at the end of your step-by-step
    breakdown in explanation.
    - If supporting_sentence_keys is an empty list, then fully_supported must be false.
    - Otherwise, use fully_supported to clarify whether everything in the response
    sentence is fully supported by the document text indicated in supporting_sentence_keys
    (fully_supported = true), or whether the sentence is only partially or incompletely
    supported by that document text (fully_supported = false).

    The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’0a.’) that
    were used to construct the answer. Include every sentence that either directly supported
    the answer, or was implicitly used to construct the answer, even if it was not used
    in its entirety. Omit sentences that were not used, and could have been removed from
    the documents without affecting the answer.

    AS A REMINDER: Your task is to review the response and assess which documents contain
    useful information pertaining to the question, and how each sentence in the response
    is supported by the text in the documents. You MUST ONLY respond with a VALID JSON string.
    Use escapes for quotes, e.g. ‘\\"‘, and newlines, e.g. ‘\\n‘.
    Do not write anything before or after the JSON string. STRICTLY Do not
    wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.

    """

    # Call the Groq API for response generation
    chat_completion = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that strictly returns JSON output."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.5,
        max_tokens=4096
    )

    # Extract the generated response
    output_text = chat_completion.choices[0].message.content.strip()

    # Try to parse the JSON response
    try:
        structured_output = json.loads(output_text)
        return structured_output
    except json.JSONDecodeError:
        return output_text




def extract_fields(output_text):
    """
    Parses the JSON response from output_text and extracts specific relevant fields.
    Always returns a dictionary, even if JSON parsing fails.
    """
    parsed_metrics = {}

    # Attempt to parse the input as JSON
    if isinstance(output_text, dict):
        # Already a dictionary, use it directly
        parsed_metrics = output_text
    elif isinstance(output_text, str):
        try:
            parsed_metrics = json.loads(output_text)
        except json.JSONDecodeError:
            # Attempt to extract JSON from a noisy string
            json_match = re.search(r"\{.*\}", output_text, re.DOTALL)
            if json_match:
                try:
                    parsed_metrics = json.loads(json_match.group(0))
                except json.JSONDecodeError:
                    print("Error: Unable to parse JSON after extraction. Returning empty dictionary.")
                    return {}
            else:
                print("Error: No valid JSON found in the input. Returning empty dictionary.")
                return {}
    else:
        print("Warning: Invalid input type. Returning empty dictionary.")
        return {}

    # Extract only the required fields
    required_fields = [
        "relevance_explanation", "all_relevant_sentence_keys",
        "overall_supported_explanation", "overall_supported",
        "sentence_support_information", "all_utilized_sentence_keys"
    ]

    extracted_data = {key: parsed_metrics.get(key, "N/A") for key in required_fields}

    return extracted_data  # Always returns a dictionary





In [ ]:
import json

def calc_rmse_scores(trace_metrics, retrieved_contexts):
    """
    Calculate the context relevance, utilization, and completeness scores
    from TRACE metrics, handling cases where values may be strings or lists.

    Parameters:
    - trace_metrics (dict or str): The TRACE metrics, containing:
        - "all_relevant_sentence_keys" (list or str): Keys of relevant sentences.
        - "all_utilized_sentence_keys" (list or str): Keys of utilized sentences.
    - retrieved_contexts (dict): A dictionary where keys are sentence keys (e.g., "0a.", "1b.")
      and values are the actual sentences in the retrieved context.

    Returns:
    - dict: A dictionary containing context relevance, utilization, and completeness scores.
    """

    # Ensure trace_metrics is a dictionary (handle string JSON input)
    if isinstance(trace_metrics, str):
        try:
            trace_metrics = json.loads(trace_metrics)
        except json.JSONDecodeError:
            print("Error: Invalid JSON format in trace_metrics. Returning empty scores.")
            return {"context_relevance_scores": {}, "utilization_scores": {}, "completeness_scores": {}}

    scores = {
        "context_relevance_scores": {},
        "utilization_scores": {},
        "completeness_scores": {},
    }

    # Initialize aggregates
    total_relevant_length = 0
    total_retrieved_length = 0
    total_utilized_length = 0
    total_relevant_for_completeness = 0

    # Extract and normalize relevant and utilized sentence keys
    all_relevant_keys = trace_metrics.get("all_relevant_sentence_keys", [])
    all_utilized_keys = trace_metrics.get("all_utilized_sentence_keys", [])

    # Convert to lists if they are strings
    if isinstance(all_relevant_keys, str):
        all_relevant_keys = all_relevant_keys.split()
    if isinstance(all_utilized_keys, str):
        all_utilized_keys = all_utilized_keys.split()

    # Ensure they are lists
    if not isinstance(all_relevant_keys, list):
        all_relevant_keys = []
    if not isinstance(all_utilized_keys, list):
        all_utilized_keys = []

    # Extract unique document keys from relevant and utilized sentences
    document_keys = set(key[0] for key in all_relevant_keys + all_utilized_keys if key)

    for document_key in document_keys:
        # Filter relevant and utilized sentences for the current document
        relevant_keys = [key for key in all_relevant_keys if key.startswith(document_key)]
        utilized_keys = [key for key in all_utilized_keys if key.startswith(document_key)]

        # Compute lengths
        relevant_length = sum(len(retrieved_contexts.get(key, "").split()) for key in relevant_keys)
        utilized_length = sum(len(retrieved_contexts.get(key, "").split()) for key in utilized_keys)
        retrieved_length = sum(len(sent.split()) for key, sent in retrieved_contexts.items() if key.startswith(document_key))

        # Context Relevance Score (per document)
        context_relevance_score = relevant_length / retrieved_length if retrieved_length > 0 else 0

        # Utilization Score (per document)
        utilization_score = utilized_length / retrieved_length if retrieved_length > 0 else 0

        # Completeness Score (per document)
        completeness_score = utilized_length / relevant_length if relevant_length > 0 else 0

        # Store individual scores for the document
        scores["context_relevance_scores"][f"document_{document_key}"] = context_relevance_score
        scores["utilization_scores"][f"document_{document_key}"] = utilization_score
        scores["completeness_scores"][f"document_{document_key}"] = completeness_score

        # Aggregate lengths for overall scores
        total_relevant_length += relevant_length
        total_retrieved_length += retrieved_length
        total_utilized_length += utilized_length
        total_relevant_for_completeness += relevant_length

    # Calculate overall scores
    overall_relevance_score = total_relevant_length / total_retrieved_length if total_retrieved_length > 0 else 0
    overall_utilization_score = total_utilized_length / total_retrieved_length if total_retrieved_length > 0 else 0
    overall_completeness_score = total_utilized_length / total_relevant_for_completeness if total_relevant_for_completeness > 0 else 0

    # Add overall scores to the result
    scores["overall_relevance_score"] = overall_relevance_score
    scores["overall_utilization_score"] = overall_utilization_score
    scores["overall_completeness_score"] = overall_completeness_score

    return scores


In [ ]:
def update_token():
    """
    Prompts the user for a new GROQ access token and updates it globally.
    """
    # Prompt for the new token securely
    new_token = getpass("Enter your new GROQ access token: ")

    # Update the environment variable
    os.environ["GROQ_TOKEN"] = new_token

    # Return the updated token
    return os.environ["GROQ_TOKEN"]


def evaluate_system(query, top_k=3, diversity=0.8):
    warnings.filterwarnings("ignore")

    # Record the start time for performance tracking
    start_time = time.time()

    # Load the embedding model
    embedding_model = load_embedding_model()

    # Retrieve top-k relevant document indices using MMR search
    selected_indices = mmr_search(query, corpus_embeddings, faiss_index, embedding_model, top_k=top_k, diversity=diversity, k_nearest_neighbors=1000)

    # Retrieve the corresponding chunks and concatenate into a single text
    retrieved_chunks = [metadata[idx] for idx in selected_indices]
    retrieved_text = " ".join(retrieved_chunks)

    # Generate a response based on the retrieved context and query
    response = generate_response(retrieved_text, query)
    response_dict = convert_responses_to_dict(response)

    # Convert chunks to dictionary with keys
    chunks_dict = convert_chunks_to_dict(retrieved_chunks)

    trace_metrics = generate_trace_metrics(chunks_dict, query, response_dict)
    metrics = extract_fields(trace_metrics)

    # Calculate RMSE scores
    rmse_score = calc_rmse_scores(metrics, chunks_dict)

    # Calculate response time
    response_time = time.time() - start_time

    # Compile the evaluation results
    return {
        "response": response,
        "response_time_seconds": response_time,
        "relevance_score": rmse_score["overall_relevance_score"],
        "utilization_score": rmse_score["overall_utilization_score"],
        "completeness_score": rmse_score["overall_completeness_score"],
        "adherence_score": metrics.get("overall_supported", "false"),
        "TRACE_metrics": trace_metrics,
        "relevance_explanation": metrics.get("relevance_explanation", " "),
        "all_relevant_sentence_keys": metrics.get("all_relevant_sentence_keys", []),
        "overall_supported_explanation": metrics.get("overall_supported_explanation", " "),
        "overall_supported": metrics.get("overall_supported", "false"),
        "sentence_support_information": metrics.get("sentence_support_information", " "),
        "all_utilized_sentence_keys": metrics.get("all_utilized_sentence_keys", []),
        "context_chunks": retrieved_chunks,
        "context_text": retrieved_text,
        "chunks_dict": chunks_dict
    }

query = "How do I fix the missing/wrong color issue ?"
evaluation = evaluate_system(query)
print(evaluation)



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'response': ' According to the text, you should ensure that the Component cables are connected to the correct jacks. This will help resolve the issue of poor color or a dim picture.', 'response_time_seconds': 4.559978246688843, 'relevance_score': 0.8099173553719008, 'utilization_score': 0.2727272727272727, 'completeness_score': 0.336734693877551, 'adherence_score': True, 'TRACE_metrics': {'relevance_explanation': 'Documents 1 and 2 contain useful information for answering the question as they provide solutions to color problems or a blank screen, which is related to the missing/wrong color issue. Document 0 is also relevant as it mentions color problems or a blank screen, but it does not provide a direct solution.', 'all_relevant_sentence_keys': ['0b.', '0c.', '1a.', '1b.', '1c.', '1d.', '2a.', '2b.', '2c.', '2d.'], 'overall_supported_explanation': 'The response claims that ensuring the Component cables are connected to the correct jacks will help resolve the issue of poor color or a 

In [ ]:
import gc


def process_split(split_name, combined_data, evaluate_system, batch_size, start_index=None, stop_index=None, max_train_samples=None):
    """
    Generic function to process a given split (train, validation, or test).

    Args:
        split_name (str): The dataset split ('train', 'validation', 'test').
        combined_data (dict): Dictionary containing all dataset splits.
        evaluate_system (callable): Function to generate responses.
        batch_size (int): Number of queries to process before writing to file.
        start_index (int, optional): Start index for the train split (default: None).
        stop_index (int, optional): Stop index for the train split (default: None).
        max_train_samples (int, optional): Max number of samples for the train split (default: None).
    """
    warnings.filterwarnings("ignore")
    base_datasets = {'emanual'}  # 'expertqa'

    if split_name not in combined_data:
        print(f"No '{split_name}' data found. Skipping.")
        return

    split_data = combined_data[split_name]
    if split_data is None or "dataset_name" not in split_data.column_names:
        print(f"'{split_name}' data does not contain dataset names. Skipping.")
        return

    all_dataset_names = set(split_data["dataset_name"])
    print(f"Available datasets in '{split_name}': {all_dataset_names}")

    for dataset in base_datasets:
        dataset_full_name = f"{dataset}_{split_name}"

        if dataset_full_name not in all_dataset_names:
            print(f"Dataset {dataset_full_name} not found in '{split_name}'. Skipping.")
            continue

        dataset_data = split_data.filter(lambda example: example["dataset_name"] == dataset_full_name)
        if len(dataset_data) == 0:
            print(f"No data found for {dataset_full_name} in '{split_name}'. Skipping.")
            continue

        # Modify train split to execute from start_index to stop_index
        if split_name == "train":
            if start_index is not None and stop_index is not None:
                dataset_data = dataset_data.select(range(start_index, stop_index))
                print(f"Selected samples from index {start_index} to {stop_index} from {dataset_full_name} (train).")
            elif max_train_samples is not None:
                num_samples = min(max_train_samples, len(dataset_data))
                dataset_data = dataset_data.select(range(num_samples))
                print(f"Selected first {num_samples} samples from {dataset_full_name} (train).")
            else:
                print(f"Processing all {len(dataset_data)} samples from {dataset_full_name} (train).")
        else:
            print(f"Processing all {len(dataset_data)} samples from {dataset_full_name} ({split_name}).")

        process_queries(dataset_data, dataset_full_name, evaluate_system, batch_size)



def process_queries(dataset_data, dataset_full_name, evaluate_system, batch_size):
    """
    Processes the queries and writes results to a CSV file in batches.

    Args:
        dataset_data: The dataset split to process.
        dataset_full_name (str): The full dataset name.
        evaluate_system (callable): Function to generate responses.
        batch_size (int): Number of queries to process before writing to file.
    """
    batch_results = []
    gc.collect()
    torch.cuda.empty_cache()

    fieldnames = [
        "dataset", "query", "query_id", "response_pred", "response_time_seconds_pred",
        "relevance_score_pred", "utilization_score_pred", "completeness_score_pred", "adherence_score_pred",
        "relevance_explanation_pred", "all_relevant_sentence_keys_pred",
        "overall_supported_explanation_pred", "overall_supported_pred",
        "sentence_support_information_pred", "all_utilized_sentence_keys_pred", "chunks_dict_pred",
        "embedding_model_pred", "annotating_model_pred", "response_generating_model_pred",
        "response_truth", "relevance_score_truth", "utilization_score_truth", "completeness_score_truth",
        "adherence_score_truth", "relevance_explanation_truth", "all_relevant_sentence_keys_truth",
        "overall_supported_explanation_truth", "overall_supported_truth",
        "sentence_support_information_truth", "all_utilized_sentence_keys_truth"
    ]

    output_file = f"{dataset_full_name}_ragbench_results.csv"
    with open(output_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    for i, example in enumerate(dataset_data):
        query = example["question"]
        print(f"Processing query {i+1}/{len(dataset_data)} from {dataset_full_name}: {query}")

        while True:  # Retry loop
            try:
                with autocast(), torch.no_grad():
                    result = evaluate_system(query)

                # If the result is successful, process it
                if result:
                    entry = {
                        "dataset": dataset_full_name,
                        "query": query,
                        "query_id": example.get("id", "N/A"),
                        "response_pred": result.get("response", ""),
                        "response_time_seconds_pred": result.get("response_time_seconds", ""),
                        "relevance_score_pred": result.get("relevance_score", ""),
                        "utilization_score_pred": result.get("utilization_score", ""),
                        "completeness_score_pred": result.get("completeness_score", ""),
                        "adherence_score_pred": result.get("adherence_score", ""),
                        "relevance_explanation_pred": result.get("relevance_explanation", ""),
                        "all_relevant_sentence_keys_pred": result.get("all_relevant_sentence_keys", ""),
                        "overall_supported_explanation_pred": result.get("overall_supported_explanation", ""),
                        "overall_supported_pred": result.get("overall_supported", ""),
                        "sentence_support_information_pred": result.get("sentence_support_information", ""),
                        "all_utilized_sentence_keys_pred": result.get("all_utilized_sentence_keys", ""),
                        "chunks_dict_pred": result.get("chunks_dict", ""),
                        "embedding_model_pred": "all-miniLM-L6-v2",
                        "annotating_model_pred": "llama3-70b-8192",
                        "response_generating_model_pred": "meta-llama/Meta-Llama-3-70B-Instruct",
                        "response_truth": example.get("response", ""),
                        "relevance_score_truth": example.get("relevance_score", ""),
                        "utilization_score_truth": example.get("utilization_score", ""),
                        "completeness_score_truth": example.get("completeness_score", ""),
                        "adherence_score_truth": example.get("adherence_score", ""),
                        "relevance_explanation_truth": example.get("relevance_explanation", ""),
                        "all_relevant_sentence_keys_truth": example.get("all_relevant_sentence_keys", ""),
                        "overall_supported_explanation_truth": example.get("overall_supported_explanation", ""),
                        "overall_supported_truth": example.get("overall_supported", ""),
                        "sentence_support_information_truth": example.get("sentence_support_information", ""),
                        "all_utilized_sentence_keys_truth": example.get("all_utilized_sentence_keys", "")
                    }

                    batch_results.append(entry)
                break

            except Exception as e:
                error_message = str(e)
                if "rate_limit_exceeded" in error_message or "Error code: 429" in error_message:
                    print("Rate limit error detected.")
                    update_token()
                    print("Token updated. Retrying...")

                else:
                    print(f"An unexpected error occurred: {e}")
                    break

        if len(batch_results) >= batch_size:
            with open(output_file, "a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerows(batch_results)
            batch_results = []
            # Clear CPU and GPU cache
            gc.collect()
            torch.cuda.empty_cache()

    if batch_results:
        with open(output_file, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerows(batch_results)
    # Clear CPU and GPU cache
    gc.collect()
    torch.cuda.empty_cache()

    print(f"All results for {dataset_full_name} saved to {output_file}")


def process_train_split(combined_data, evaluate_system, batch_size=5, start_index=0, stop_index=1054):
    """Processes the train split ."""
    process_split("train", combined_data, evaluate_system, batch_size, start_index=start_index, stop_index=stop_index)

def process_validation_split(combined_data, evaluate_system, batch_size=5):
    """Processes the validation split (all samples)."""
    process_split("validation", combined_data, evaluate_system, batch_size)

def process_test_split(combined_data, evaluate_system, batch_size=5):
    """Processes the test split (all samples)."""
    process_split("test", combined_data, evaluate_system, batch_size)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:

gc.collect()
torch.cuda.empty_cache()
process_train_split(combined_data, evaluate_system, batch_size=5)


Available datasets in 'train': {'delucionqa_train', 'pubmedqa_train', 'covidqa_train', 'tatqa_train', 'hotpotqa_train', None, 'expertqa_train', 'emanual_train', 'msmarco_train', 'cuad_train', 'hagrid_train'}
Selected samples from index 0 to 1054 from emanual_train (train).
Processing query 1/1054 from emanual_train: How do I select Natural mode?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 2/1054 from emanual_train: What are the steps to connect an IP control device to the TV?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 3/1054 from emanual_train: Can I update TV’s software automatically?


...

Processing query 1053/1054 from emanual_train: How do I check the Internet connection status?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 1054/1054 from emanual_train: Where can I select the services which I want to be notified?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


All results for emanual_train saved to emanual_train_ragbench_results.csv


In [ ]:

gc.collect()
torch.cuda.empty_cache()
process_validation_split(combined_data, evaluate_system, batch_size=5)


Available datasets in 'validation': {'hagrid_validation', 'delucionqa_validation', 'emanual_validation', 'expertqa_validation', 'cuad_validation', 'hotpotqa_validation', 'covidqa_validation', None, 'msmarco_validation', 'tatqa_validation', 'pubmedqa_validation'}


Filter:   0%|          | 0/8223 [00:00<?, ? examples/s]

Processing all 132 samples from emanual_validation (validation).
Processing query 1/132 from emanual_validation: What is the use of search option in tne smart hub?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 2/132 from emanual_validation: Can I select Sound Feedback?


...

Processing query 131/132 from emanual_validation:  I don't know about Ambient Mode. Can you explain briefly?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 132/132 from emanual_validation: How to update TV's software through internet?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


All results for emanual_validation saved to emanual_validation_ragbench_results.csv


In [ ]:

gc.collect()
torch.cuda.empty_cache()
process_test_split(combined_data, evaluate_system, batch_size=5)


Available datasets in 'test': {'cuad_test', 'covidqa_test', 'expertqa_test', 'hagrid_test', 'hotpotqa_test', 'delucionqa_test', 'tatqa_test', 'emanual_test', 'pubmedqa_test', 'msmarco_test'}
Processing all 132 samples from emanual_test (test).
Processing query 1/132 from emanual_test: I want to  enter into Ambient mode. How can I do that?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 2/132 from emanual_test: Where do I find signal information ?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 3/132 from emanual_test: How can I view the channels that are serached by auto program function and How can I view first five favorite channel?


...

Processing query 131/132 from emanual_test: How do I enable/disable light effect?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing query 132/132 from emanual_test: How to create new account in SmartThings ?


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


All results for emanual_test saved to emanual_test_ragbench_results.csv


In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, roc_auc_score, f1_score, precision_score, accuracy_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

# Load the Excel file
file_path = "COMBINED EMANUAL RESULTS.xlsx"

# Read all sheet names
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names[:15]

# Define columns of interest
columns = [
    "relevance_score_pred", "utilization_score_pred", "completeness_score_pred", "adherence_score_pred",
    "relevance_score_truth", "utilization_score_truth", "completeness_score_truth", "adherence_score_truth"
]

# Function to compute RMSE scores for regression fields
def compute_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Function to compute classification metrics for adherence scores
def compute_classification_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    auc_roc = roc_auc_score(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0  # True Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0  # False Positive Rate

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "auc_roc": auc_roc,
        "TPR": tpr,
        "FPR": fpr
    }

# Dataframe to store results
metrics_data = []

# Iterate through each sheet and compute metrics
for sheet in sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet, usecols=columns)
    df = df.fillna(0)  # Handle NaN values

    # Compute RMSE metrics
    rmse_results = {
        "relevance_rmse": compute_rmse(df["relevance_score_truth"], df["relevance_score_pred"]),
        "utilization_rmse": compute_rmse(df["utilization_score_truth"], df["utilization_score_pred"]),
        "completeness_rmse": compute_rmse(df["completeness_score_truth"], df["completeness_score_pred"]),
    }

    # Compute classification metrics
    classification_metrics = compute_classification_metrics(
        df["adherence_score_truth"], df["adherence_score_pred"]
    )

    # Store results in list
    metrics_data.append({
        "Sheet Name": sheet,
        **rmse_results,
        **classification_metrics
    })

# Convert results to DataFrame
metrics_df = pd.DataFrame(metrics_data)

# Save results to a new sheet named "METRICS"
with pd.ExcelWriter(file_path, engine="openpyxl", mode="a") as writer:
    metrics_df.to_excel(writer, sheet_name="METRICS", index=False)

print("Metrics successfully saved in 'METRICS' sheet of the same Excel file.")


Metrics successfully saved in 'METRICS' sheet of the same Excel file.


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

# Load the Excel file
file_path = "COMBINED EMANUAL RESULTS.xlsx"

# Read metrics from "METRICS" sheet
metrics_df = pd.read_excel(file_path, sheet_name="METRICS")

# Function to visualize metrics separately for Train, Validation, and Test and save to PDF
def visualize_metrics(metrics_df, split_type, pdf_pages):
    # Filter data for the given split type
    split_df = metrics_df[metrics_df["Sheet Name"].str.contains(split_type, case=False)]
    if split_df.empty:
        print(f"No data available for {split_type} split.")
        return

    # Set plot style
    sns.set(style="white")

    # Define light colors for visualization
    light_colors = sns.color_palette("pastel", n_colors=6)

    # Bar plots for RMSE metrics
    rmse_metrics = ["relevance_rmse", "utilization_rmse", "completeness_rmse"]
    plt.figure(figsize=(12, 6))
    ax = split_df.plot(x="Sheet Name", y=rmse_metrics, kind="bar", color=light_colors,
                       title=f"RMSE Comparison ({split_type} Split)", ax=plt.gca())

    # Show number values on bars
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=10, color='black', fontweight='bold', xytext=(0, 8), textcoords='offset points')

    plt.xticks(rotation=45)
    plt.ylabel("RMSE Value", fontweight='bold', fontsize=12)
    plt.legend(title="Metrics", loc='upper left', bbox_to_anchor=(1, 1), title_fontsize=12, fontsize=10)
    plt.title(f'RMSE Comparison ({split_type} Split)', fontsize=14, fontweight='bold')
    ax.grid(False)  # Remove gridlines
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # Bar plots for Classification Metrics
    class_metrics = ["accuracy", "precision", "recall", "f1_score"]
    plt.figure(figsize=(12, 6))
    ax = split_df.plot(x="Sheet Name", y=class_metrics, kind="bar", color=light_colors,
                       title=f"Classification Metrics Comparison ({split_type} Split)", ax=plt.gca())

    # Show number values on bars
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=10, color='black', fontweight='bold', xytext=(0, 8), textcoords='offset points')

    plt.xticks(rotation=45)
    plt.ylabel("Score", fontweight='bold', fontsize=12)
    plt.legend(title="Metrics", loc='upper left', bbox_to_anchor=(1, 1), title_fontsize=12, fontsize=10)
    plt.title(f'Classification Metrics Comparison ({split_type} Split)', fontsize=14, fontweight='bold')
    ax.grid(False)  # Remove gridlines
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # ROC Curve
    plt.figure(figsize=(8, 6))
    for index, row in split_df.iterrows():
        fpr = [0, row["FPR"], 1]
        tpr = [0, row["TPR"], 1]
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{row["Sheet Name"]} (AUC = {roc_auc:.2f})', color=light_colors[index % len(light_colors)])

    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
    plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
    plt.title(f'ROC Curves for Adherence Score ({split_type} Split)', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10, title_fontsize=12, title="Models", bbox_to_anchor=(1.05, 0.5))
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # Confusion Matrix Heatmap for each sheet in the split
    for index, row in split_df.iterrows():
        cm = np.array([[1 - row["FPR"], row["FPR"]], [1 - row["TPR"], row["TPR"]]])
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"], cbar=False)
        plt.xlabel("Predicted", fontweight='bold', fontsize=12)
        plt.ylabel("Actual", fontweight='bold', fontsize=12)
        plt.title(f'Confusion Matrix - {row["Sheet Name"]} ({split_type} Split)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf_pages.savefig()
        plt.close()

# Create a PDF file to save all the visualizations
pdf_filename = "RAG_Metrics_Visualization by Splits.pdf"
with PdfPages(pdf_filename) as pdf_pages:
    for split in ["Train", "Val", "Test"]:
        visualize_metrics(metrics_df, split, pdf_pages)

print(f"Visualization saved successfully to {pdf_filename}.")


Visualization saved successfully to RAG_Metrics_Visualization by Splits.pdf.


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

# Load the Excel file
file_path = "COMBINED EMANUAL RESULTS.xlsx"

# Read metrics from "METRICS" sheet
metrics_df = pd.read_excel(file_path, sheet_name="METRICS")

# Combine Train, Test, and Validation splits for each model
metrics_df["Model"] = metrics_df["Sheet Name"].str.extract(r'(Model-\d)')

# Keep only numeric columns before aggregation to avoid dtype errors
numeric_columns = metrics_df.select_dtypes(include=[np.number]).columns
combined_metrics = metrics_df[numeric_columns].groupby(metrics_df["Model"]).mean().reset_index()

# Set plot style
sns.set(style="white")

# Define light colors for visualization
light_colors = sns.color_palette("pastel", n_colors=6)

# Create a PDF file to save all the visualizations
pdf_filename = "RAG_Metrics_Visualization_BY_Model.pdf"
with PdfPages(pdf_filename) as pdf_pages:

    # Combined Bar Plots for RMSE metrics across models
    rmse_metrics = ["relevance_rmse", "utilization_rmse", "completeness_rmse"]
    plt.figure(figsize=(12, 6))
    ax = combined_metrics.plot(x="Model", y=rmse_metrics, kind="bar", color=light_colors, title="RMSE Comparison Across Models", ax=plt.gca())

    # Show number values on bars
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=10, color='black', fontweight='bold', xytext=(0, 8), textcoords='offset points')

    plt.xticks(rotation=45)
    plt.ylabel("RMSE Value", fontweight='bold', fontsize=12)
    plt.legend(title="Metrics", loc='upper left', bbox_to_anchor=(1, 1), title_fontsize=12, fontsize=10)
    plt.title('RMSE Comparison Across Models', fontsize=14, fontweight='bold')
    ax.grid(False)  # Remove gridlines
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # Combined Bar Plots for Classification Metrics across models
    class_metrics = ["accuracy", "precision", "recall", "f1_score"]
    plt.figure(figsize=(12, 6))
    ax = combined_metrics.plot(x="Model", y=class_metrics, kind="bar", color=light_colors, title="Classification Metrics Comparison Across Models", ax=plt.gca())

    # Show number values on bars
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=10, color='black', fontweight='bold', xytext=(0, 8), textcoords='offset points')

    plt.xticks(rotation=45)
    plt.ylabel("Score", fontweight='bold', fontsize=12)
    plt.legend(title="Metrics", loc='upper left', bbox_to_anchor=(1, 1), title_fontsize=12, fontsize=10)
    plt.title('Classification Metrics Comparison Across Models', fontsize=14, fontweight='bold')
    ax.grid(False)  # Remove gridlines
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # Combined ROC Curve
    plt.figure(figsize=(8, 6))
    for index, row in combined_metrics.iterrows():
        fpr = [0, row["FPR"], 1]
        tpr = [0, row["TPR"], 1]
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{row["Model"]} (AUC = {roc_auc:.2f})', color=light_colors[index % len(light_colors)])

    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
    plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
    plt.title('Combined ROC Curves for Adherence Score Across Models', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10, title_fontsize=12, title="Models", bbox_to_anchor=(1.05, 0.5))
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()

    # Combined Confusion Matrix Heatmap for each model
    for index, row in combined_metrics.iterrows():
        cm = np.array([[1 - row["FPR"], row["FPR"]], [1 - row["TPR"], row["TPR"]]])
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"], cbar=False)
        plt.xlabel("Predicted", fontweight='bold', fontsize=12)
        plt.ylabel("Actual", fontweight='bold', fontsize=12)
        plt.title(f'Confusion Matrix - {row["Model"]}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf_pages.savefig()
        plt.close()

print(f"Visualization saved successfully to {pdf_filename}.")


Visualization saved successfully to RAG_Metrics_Visualization.pdf.


In [ ]:

# Define Gradio interface
def gradio_interface(query, top_k=3, diversity=0.7):
    evaluation = evaluate_system(query=query, top_k=3, diversity=0.7)
    if evaluation and isinstance(evaluation, dict):
        relevance = evaluation.get("relevance_score", "N/A")
        utilization = evaluation.get("utilization_score", "N/A")
        completeness = evaluation.get("completeness_score", "N/A")
        adherence = evaluation.get("adherence_score", "N/A")
    else:
        relevance, utilization, completeness, adherence = "N/A", "N/A", "N/A", "false"

    return f"Response: {evaluation['response']}\n\nRelevance: {relevance}\nUtilization: {utilization}\nCompleteness: {completeness}\nAdherence: {adherence}\nResponse Time: {evaluation['response_time_seconds']}\n\nRelevance Explanation: {evaluation['relevance_explanation']}\nAll Relevant Sentence Keys:{evaluation['all_relevant_sentence_keys']}\nOverall Supported Explanation: {evaluation['overall_supported_explanation']}\nOverall Supported: {evaluation['overall_supported']}\nSentence Support Information: {evaluation['sentence_support_information']}\nAll Utilized Sentence Keys: {evaluation['all_utilized_sentence_keys']}"


interface = gr.Interface(
    fn=gradio_interface,
    inputs="text",
    outputs="text",
    title="RAG System Evaluation",
    description="Ask questions and get responses from RAG system."
)
interface.launch()


'\n# Define Gradio interface\ndef gradio_interface(query, top_k=3, diversity=0.7):\n    evaluation = evaluate_system(query=query, top_k=3, diversity=0.7)\n    if evaluation and isinstance(evaluation, dict):\n        relevance = evaluation.get("relevance_score", "N/A")\n        utilization = evaluation.get("utilization_score", "N/A")\n        completeness = evaluation.get("completeness_score", "N/A")\n        adherence = evaluation.get("adherence_score", "N/A")\n    else:\n        relevance, utilization, completeness, adherence = "N/A", "N/A", "N/A", "false"\n\n    return f"Response: {evaluation[\'response\']}\n\nRelevance: {relevance}\nUtilization: {utilization}\nCompleteness: {completeness}\nAdherence: {adherence}\nResponse Time: {evaluation[\'response_time_seconds\']}\n\nRelevance Explanation: {evaluation[\'relevance_explanation\']}\nAll Relevant Sentence Keys:{evaluation[\'all_relevant_sentence_keys\']}\nOverall Supported Explanation: {evaluation[\'overall_supported_explanation\']}\